# TRIBE V2 — Simple pre-render

**Before running:** Runtime → Change runtime type → T4 GPU

Run cells top to bottom. Each video takes ~5–10 min on T4.

In [ ]:
# 1. Confirm GPU
!nvidia-smi

In [ ]:
# 2. Install dependencies (~3 min) — kernel will auto-restart after this cell
# numpy must be pinned to <2.0 — tribev2 uses internal APIs removed in numpy 2.x
!pip install -q "numpy<2.0"
!pip install -q "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
!pip install -q nilearn nibabel
import os
os.kill(os.getpid(), 9)  # auto-restart so numpy downgrade takes effect

In [ ]:
# 3. HuggingFace login
# Needs read access to facebook/tribev2 and meta-llama/Llama-3.2-3B (gated)
# Get token at: https://huggingface.co/settings/tokens
from huggingface_hub import login
login()

In [ ]:
# 4. Upload aggregate.py and atlas.py
# aggregate.py -> archive/tribe-pipeline/aggregate.py
# atlas.py     -> backend/atlas.py
# Both must land in /content/ so the subprocess in cell 6 can find them.
import os
os.chdir('/content')
from google.colab import files
print(f'cwd = {os.getcwd()}')
print('Upload BOTH aggregate.py (from archive/tribe-pipeline/) and atlas.py (from backend/)')
files.upload()
assert os.path.exists('/content/aggregate.py'), 'aggregate.py missing from /content/'
assert os.path.exists('/content/atlas.py'),     'atlas.py missing from /content/'
print('OK - both scripts present in /content/')

In [ ]:
# 5. Upload all your video files at once
import os
os.chdir('/content')
from google.colab import files
print('Upload your video files (.mp4 etc) - you can select multiple')
uploaded = files.upload()
# Force absolute paths so cwd shifts later in the notebook can't break lookups.
video_paths = [os.path.join('/content', name) for name in uploaded.keys()]
print(f'Uploaded {len(video_paths)} video(s):')
for v in video_paths:
    assert os.path.exists(v), f'missing: {v}'
    print(f'  {v}')

In [ ]:
# 6. Run TRIBE V2 inference on all uploaded videos
# Writes outputs to /content/prerendered/<stem>/activity.json so the structure
# matches backend/prerendered/ in the repo. Drop the unzipped folder straight in.
import os, sys, subprocess, pickle, json
from pathlib import Path
import numpy as np
from tribev2.demo_utils import TribeModel

os.chdir('/content')
ROOT    = Path('/content')
CACHE   = ROOT / 'cache'
OUT_DIR = ROOT / 'prerendered'
CACHE.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

print('Loading model...')
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=CACHE)

for video_path in video_paths:
    video_path = str(video_path)              # absolute /content/<file>
    stem = Path(video_path).stem
    clip_dir = OUT_DIR / stem
    clip_dir.mkdir(parents=True, exist_ok=True)

    preds_npz     = clip_dir / 'preds.npz'
    segments_pkl  = clip_dir / 'preds.segments.pkl'
    activity_json = clip_dir / 'activity.json'

    print(f'\n==> [{stem}] Running inference...')
    df = model.get_events_dataframe(video_path=video_path)
    preds, segments = model.predict(events=df)
    print(f'    preds shape: {preds.shape}')

    np.savez(preds_npz, preds=preds.astype('float32'))
    with open(segments_pkl, 'wb') as f:
        pickle.dump(segments, f)

    print(f'    Aggregating -> {activity_json}')
    result = subprocess.run(
        [sys.executable, '/content/aggregate.py',
         '--preds',    str(preds_npz),
         '--segments', str(segments_pkl),
         '--out',      str(activity_json),
         '--atlas',    'yeo7',
         '--cache',    str(CACHE)],
        capture_output=True, text=True, cwd='/content',
    )
    # Always print both streams so silent failures are visible.
    if result.stdout: print('    [stdout]', result.stdout.strip())
    if result.stderr: print('    [stderr]', result.stderr.strip())
    if result.returncode != 0:
        print(f'    *** aggregate.py FAILED (rc={result.returncode}) for {stem} ***')
        continue
    if not activity_json.exists():
        print(f'    *** activity.json missing for {stem} ***')
        continue
    with open(activity_json) as f:
        d = json.load(f)
    print(f'    Done: {d["n_timesteps"]} frames -> {activity_json}')

print('\nAll videos processed.')
print('Output tree:')
subprocess.run(['ls', '-laR', str(OUT_DIR)])

In [ ]:
# 7. Zip the whole prerendered/ tree and download it.
# files.download() can't download a directory, so we zip first.
# Unzip locally and drop the folders into backend/prerendered/.
import os, shutil
from pathlib import Path
from google.colab import files

OUT_DIR = Path('/content/prerendered')
clips = sorted([p.name for p in OUT_DIR.iterdir() if p.is_dir() and (p / 'activity.json').exists()])
if not clips:
    raise RuntimeError(f'No activity.json found under {OUT_DIR}. Re-run cell 6 and check stderr.')

print(f'Packaging {len(clips)} clip(s):')
for c in clips:
    print(f'  {c}/activity.json')

archive = shutil.make_archive('/content/prerendered', 'zip', root_dir='/content', base_dir='prerendered')
print(f'Wrote {archive} ({os.path.getsize(archive)/1024:.1f} KB)')
files.download(archive)
print('Unzip locally, then move each <clip>/activity.json into backend/prerendered/<clip>/')